In [116]:
import pickle
import zipfile
import numpy as np
import pandas as pd
import os
from scipy import stats

In [ ]:
#Set directory with WESAD data here
wesad_dir = Path.home() / "Downloads" / "WESAD"
#Empty list to store all the participants
subject_list = []
#Loop to open all the files
for i in wesad_dir.rglob("*.pkl"):
    subject_id = i.parent.name
    with open(i, "rb") as file:
        data = pickle.load(file, encoding='latin1')
        wrist_data = data['signal']['wrist']
        eda_signal = wrist_data['EDA'].flatten()
        #APPARENTLY the label is for 700hz and this needs to be recounted to match labels to 4hz accordingly
        label_700hz = data['label'].flatten()
        #700/4=175 so we take chunks every 175 then label based on mode rule, I am open to suggestions for downsampling the labels
        n_chunks = len(label_700hz) // 175
        label_chunks = label_700hz[: n_chunks * 175].reshape(n_chunks, 175)
        label_4hz = stats.mode(label_chunks, axis=1).mode.flatten()
        #Rounding issue with getting mismatched vector lengths for concatenation
        min_len = min(len(eda_signal), len(label_4hz))
        #Build Dataframe
        df_subject = pd.DataFrame({
            'EDA': eda_signal[:min_len],
            'Label': label_4hz[:min_len]
            })
        df_subject['Subject'] = subject_id
        subject_list.append(df_subject)
#Check dataframe was built correctly        
final_df = pd.concat(subject_list, ignore_index=True)
print(final_df.head())

        EDA  Label Subject
0  0.220298      0     S15
1  0.170407      0     S15
2  0.189596      0     S15
3  0.151218      0     S15
4  0.166569      0     S15


In [118]:
final_df['Subject'].unique()

array(['S15', 'S7', 'S3', 'S2', 'S14', 'S8', 'S11', 'S17', 'S10', 'S16',
       'S4', 'S13', 'S5', 'S9', 'S6'], dtype=object)

In [119]:
#Relabel numbers for easy human readable labels
final_df.loc[final_df['Label'] == 0, 'Label'] = 'Transient'
final_df.loc[final_df['Label'] == 1, 'Label'] = 'Baseline'
final_df.loc[final_df['Label'] == 2, 'Label'] = 'Stress'
final_df.loc[final_df['Label'] == 3, 'Label'] = 'Amusement'
final_df.loc[final_df['Label'] == 4, 'Label'] = 'Meditation'
final_df.loc[final_df['Label'] == 5, 'Label'] = 'Ignore'
final_df.loc[final_df['Label'] == 6, 'Label'] = 'Ignore'
final_df.loc[final_df['Label'] == 7, 'Label'] = 'Ignore'
final_df['Label'].unique()

/tmp/ipykernel_54950/166819567.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'Transient' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[final_df['Label'] == 0, 'Label'] = 'Transient'


array(['Transient', 'Baseline', 'Ignore', 'Amusement', 'Meditation',
       'Stress'], dtype=object)

In [120]:
#Removed noise so it does not affect the decomposition stages
final_df = final_df[~final_df['Label'].isin(['Transient', 'Ignore'])]
final_df['Label'].unique()

array(['Baseline', 'Amusement', 'Meditation', 'Stress'], dtype=object)

In [121]:
#Store all participants IDs
subject_records = final_df['Subject'].unique()
df_copy = final_df.copy()
#Store a dictionary of all the subjects and their unique timeline (unique as in randomly rotated two different designs and I would rather store their individual labels this)
new_data = {}
for i in subject_records:
    participant_data = df_copy.loc[df_copy['Subject'] == i, ['EDA', 'Label']].reset_index(drop=True)
    new_data[f'{i}_eda'] = participant_data['EDA']
    new_data[f'{i}_label'] = participant_data['Label']
signal_column_df = pd.DataFrame(new_data)
signal_column_df    

,S15_eda,S15_label,S7_eda,S7_label,S3_eda,S3_label,S2_eda,S2_label,S14_eda,S14_label,...,S4_eda,S4_label,S13_eda,S13_label,S5_eda,S5_label,S9_eda,S9_label,S6_eda,S6_label
0,0.257396,Baseline,6.156929,Baseline,6.701712,Baseline,1.645664,Baseline,0.303449,Baseline,...,0.224122,Baseline,8.959934,Baseline,1.357394,Baseline,0.419836,Baseline,3.007215,Baseline
1,0.254837,Baseline,6.146695,Baseline,6.553087,Baseline,1.640539,Baseline,0.295773,Baseline,...,0.222843,Baseline,8.885738,Baseline,1.356115,Baseline,0.422394,Baseline,2.990589,Baseline
2,0.257396,Baseline,6.122389,Baseline,6.556931,Baseline,1.634132,Baseline,0.291936,Baseline,...,0.222843,Baseline,8.837127,Baseline,1.359951,Baseline,0.422394,Baseline,3.017447,Baseline
3,0.248441,Baseline,6.080174,Baseline,6.587681,Baseline,1.614912,Baseline,0.293215,Baseline,...,0.224122,Baseline,8.807704,Baseline,1.359951,Baseline,0.422394,Baseline,3.008494,Baseline
4,0.247162,Baseline,6.084012,Baseline,6.601775,Baseline,1.591848,Baseline,0.285539,Baseline,...,0.225401,Baseline,8.879342,Baseline,1.358673,Baseline,0.424952,Baseline,3.017447,Baseline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12287,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12288,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#We should have a function here to handle the decompostion of the full signals here before doing reshaping

In [ ]:
#Now we make the reshape function
#4 parameters, your dataframe, your window size or how many hz of rows should be in each row, purity for how much each row can tolerate label confounsion, and stride size for how much we should overlap
#By default this is every 10 seconds of data
def reshape_function(df, window_size = 10, stride_size = 10, hertz=4, purity_threshold=0.8):
    window_row = window_size*hertz
    stride_row = stride_size*hertz
    #Get the subject names
    subjects = sorted(
        list(set(col.split("_")[0] for col in df.columns if "_" in col))
    )
    #Store the chunks of data while we're looping across the time span
    all_window = []
    for i in subjects:
        eda_col = f'{i}_eda'
        label_col = f'{i}_label'
        #Kill NAs in individual sequence
        individual_df = df[[eda_col,label_col]].dropna()
        total_rows = len(individual_df)
        #For each subject we take a chunk of their data by our determined window size then stack onto a list (in this case "all window")
        #Chunk is determined through standard range check then we check the purity threshold if we keep the chunk, based on birken's paper design
        for j in range(0,total_rows-window_row+1, stride_row):
            end = j + window_row
            signal_window = individual_df[eda_col].iloc[j:end].values
            label_window = individual_df[label_col].iloc[j:end].values

            # Determine the majority label and calculate if there exists a supermajority in the chunk to determine if kept
            value_counts = pd.Series(label_window).value_counts()
            majority_label = value_counts.index[0]
            majority_count = value_counts.iloc[0]

            purity = majority_count / window_row

            # Retain only windows meeting or exceeding the requested purity ratio
            if purity >= purity_threshold:
                row_data = [i, majority_label] + list(signal_window)
                all_window.append(row_data)

    feature_cols = [f"sample_{i}" for i in range(window_row)]
    columns = ["subject_id", "label"] + feature_cols

    return pd.DataFrame(all_window, columns=columns)

    





In [123]:
reshape_function(signal_column_df)

,subject_id,label,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_30,sample_31,sample_32,sample_33,sample_34,sample_35,sample_36,sample_37,sample_38,sample_39
0,S10,Baseline,0.377356,0.373519,0.374798,0.373519,0.376077,0.372240,0.364565,0.367123,...,0.377356,0.373519,0.368402,0.367123,0.373519,0.395265,0.373519,0.376077,0.378636,0.379915
1,S10,Baseline,0.378636,0.376077,0.376077,0.382473,0.401661,0.387590,0.386311,0.382473,...,0.374798,0.405498,0.399102,0.392707,0.388869,0.387590,0.381194,0.382473,0.378636,0.516786
2,S10,Baseline,0.607608,0.589699,0.571791,0.543649,0.520624,0.505274,0.492482,0.478411,...,0.423407,0.427244,0.428523,0.427244,0.427244,0.425965,0.425965,0.425965,0.424686,0.423407
3,S10,Baseline,0.423407,0.424686,0.418290,0.423407,0.420848,0.422128,0.423407,0.424686,...,0.374798,0.379915,0.373519,0.372240,0.370961,0.370961,0.369681,0.378636,0.382473,0.382473
4,S10,Baseline,0.382473,0.381194,0.382473,0.381194,0.386311,0.390148,0.383752,0.382473,...,0.378636,0.377356,0.377356,0.377356,0.377356,0.377356,0.377356,0.378636,0.377356,0.374798
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4460,S9,Meditation,0.395531,0.392973,0.394252,0.394252,0.394252,0.389135,0.394252,0.390415,...,0.395531,0.395531,0.392973,0.394252,0.395531,0.392973,0.392973,0.394252,0.394252,0.394252
4461,S9,Meditation,0.392973,0.394252,0.394252,0.395531,0.392973,0.392973,0.391694,0.394252,...,0.391694,0.392973,0.392973,0.392973,0.392973,0.391694,0.391694,0.390415,0.392973,0.389135
4462,S9,Meditation,0.392973,0.390415,0.392973,0.391694,0.391694,0.392973,0.391694,0.385298,...,0.390415,0.390415,0.386577,0.390415,0.389135,0.391694,0.384019,0.391694,0.391694,0.390415
4463,S9,Meditation,0.391694,0.391694,0.391694,0.389135,0.390415,0.387856,0.392973,0.385298,...,0.391694,0.389135,0.391694,0.385298,0.390415,0.390415,0.387856,0.389135,0.387856,0.389135
